In [1]:
import requests
import pandas as pd
import os

# Set up the folder paths
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [2]:
# Base URL for the UKHSA API
base_url = ("https://api.ukhsa-dashboard.data.gov.uk/themes/medicines"
            "/sub_themes/antimicrobials/topics/antibiotics"
            "/geography_types/Nation/geographies/England/metrics/")

# The specific metrics we want to download
metrics = {
    'total_antibiotic_use': 'antibiotics_consumption_totalDID',
    'primary_care': 'antibiotics_consumption_primaryDID',
    'secondary_care': 'antibiotics_consumption_secondaryDID',
    'access_antibiotics': 'antibiotics_consumption_totalAware' 
}

print("URLs constructed successfully.")

URLs constructed successfully.


In [3]:
# Fetch Total Antibiotic Use
url = base_url + metrics['total_antibiotic_use']
response = requests.get(url, params={'page_size': 1000})

# Check if the request was successful
print("Status Code:", response.status_code)

# Convert to DataFrame and look at the first few rows
total_data = response.json()
df_total = pd.DataFrame(total_data['results'])
df_total.head()

Status Code: 200


,theme,sub_theme,topic,geography_type,geography,geography_code,metric,metric_group,stratum,sex,age,year,month,epiweek,date,metric_value,in_reporting_delay_period
0,medicines,antimicrobials,antibiotics,Nation,England,E92000001,antibiotics_consumption_totalDID,consumption,default,all,all,2016,4,17,2016-04-30,3.32,False
1,medicines,antimicrobials,antibiotics,Nation,England,E92000001,antibiotics_consumption_totalDID,consumption,default,all,all,2016,5,22,2016-05-31,3.20,False
2,medicines,antimicrobials,antibiotics,Nation,England,E92000001,antibiotics_consumption_totalDID,consumption,default,all,all,2016,6,26,2016-06-30,3.33,False
3,medicines,antimicrobials,antibiotics,Nation,England,E92000001,antibiotics_consumption_totalDID,consumption,default,all,all,2016,7,30,2016-07-31,3.17,False
4,medicines,antimicrobials,antibiotics,Nation,England,E92000001,antibiotics_consumption_totalDID,consumption,default,all,all,2016,8,35,2016-08-31,3.27,False


In [4]:
# Fetch Primary Care
url = base_url + metrics['primary_care']
df_primary = pd.DataFrame(requests.get(url, params={'page_size': 1000}).json()['results'])

# Fetch Secondary Care
url = base_url + metrics['secondary_care']
df_secondary = pd.DataFrame(requests.get(url, params={'page_size': 1000}).json()['results'])

# Check the columns of the primary care data
print(df_primary.columns.tolist())
df_primary.head(3)

['theme', 'sub_theme', 'topic', 'geography_type', 'geography', 'geography_code', 'metric', 'metric_group', 'stratum', 'sex', 'age', 'year', 'month', 'epiweek', 'date', 'metric_value', 'in_reporting_delay_period']


,theme,sub_theme,topic,geography_type,geography,geography_code,metric,metric_group,stratum,sex,age,year,month,epiweek,date,metric_value,in_reporting_delay_period
0,medicines,antimicrobials,antibiotics,Nation,England,E92000001,antibiotics_consumption_primaryDID,consumption,default,all,all,2016,4,17,2016-04-30,0.0,False
1,medicines,antimicrobials,antibiotics,Nation,England,E92000001,antibiotics_consumption_primaryDID,consumption,default,all,all,2016,5,22,2016-05-31,0.0,False
2,medicines,antimicrobials,antibiotics,Nation,England,E92000001,antibiotics_consumption_primaryDID,consumption,default,all,all,2016,6,26,2016-06-30,0.0,False


In [5]:
# Fetch AWaRe data
url = base_url + metrics['access_antibiotics']
df_aware_raw = pd.DataFrame(requests.get(url, params={'page_size': 1000}).json()['results'])

# Look at the unique strata (Access, Watch, Reserve)
print(df_aware_raw['stratum'].unique())

['dids_a' 'dids_w' 'dids_r']


In [6]:
# 1. Clean the Total data (keep date and metric_value)
df_total_clean = df_total[['date', 'metric_value']].rename(columns={'metric_value': 'total_did'})

# 2. Clean Primary and Secondary
df_primary_clean = df_primary[['date', 'metric_value']].rename(columns={'metric_value': 'primary_care_did'})
df_secondary_clean = df_secondary[['date', 'metric_value']].rename(columns={'metric_value': 'secondary_care_did'})

# 3. Extract Access, Watch, and Reserve from the AWaRe data
df_access = df_aware_raw[df_aware_raw['stratum'] == 'dids_a'][['date', 'metric_value']].rename(columns={'metric_value': 'access_did'})
df_watch = df_aware_raw[df_aware_raw['stratum'] == 'dids_w'][['date', 'metric_value']].rename(columns={'metric_value': 'watch_did'})
df_reserve = df_aware_raw[df_aware_raw['stratum'] == 'dids_r'][['date', 'metric_value']].rename(columns={'metric_value': 'reserve_did'})

print("Data cleaned and columns renamed.")

Data cleaned and columns renamed.


In [7]:
# Merge all the cleaned dataframes on the 'date' column
merged_df = (df_total_clean
             .merge(df_primary_clean, on='date', how='outer')
             .merge(df_secondary_clean, on='date', how='outer')
             .merge(df_access, on='date', how='outer')
             .merge(df_watch, on='date', how='outer')
             .merge(df_reserve, on='date', how='outer'))

# Convert the date column to a proper datetime format and sort it
merged_df['date'] = pd.to_datetime(merged_df['date'])
merged_df = merged_df.sort_values('date').reset_index(drop=True)

# Show the final result
print("Final shape:", merged_df.shape)
merged_df.head()

Final shape: (117, 7)


,date,total_did,primary_care_did,secondary_care_did,access_did,watch_did,reserve_did
0,2016-04-30,3.32,0.0,3.32,NaN,NaN,NaN
1,2016-05-31,3.20,0.0,3.20,NaN,NaN,NaN
2,2016-06-30,3.33,0.0,3.33,NaN,NaN,NaN
3,2016-07-31,3.17,0.0,3.17,NaN,NaN,NaN
4,2016-08-31,3.27,0.0,3.27,NaN,NaN,NaN


In [8]:
# Save to the processed folder
output_path = os.path.join(PROCESSED_DIR, 'ukhsa_consumption_clean.csv')
merged_df.to_csv(output_path, index=False)

print(f"Data successfully saved to {output_path}")

Data successfully saved to data/processed/ukhsa_consumption_clean.csv
